In [ ]:
!pip install -q ultralytics

import os
import shutil
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

print("Ultralytics:", __import__("ultralytics").__version__)

In [ ]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

# CHANGE THIS to the location of your original dataset in Kaggle
SOURCE_ROOT = Path("/kaggle/input/YOUR_DATASET_FOLDER")

WORK_ROOT = Path("/kaggle/working")

BINARY_ROOT = WORK_ROOT / "railway_defect_binary"
RESULTS_ROOT = WORK_ROOT / "railway_yolo11"

MODEL_OUTPUT = WORK_ROOT / "YOLO11_railway_defect_best.pt"

print("Source dataset:", SOURCE_ROOT)
print("Binary dataset:", BINARY_ROOT)
print("Results:", RESULTS_ROOT)

In [ ]:
# ============================================================
# CHECK DATASET STRUCTURE
# ============================================================

for split in ["train", "valid", "test"]:
    images = SOURCE_ROOT / split / "images"
    labels = SOURCE_ROOT / split / "labels"

    print(f"\n{split.upper()}")
    print("Images exists:", images.exists())
    print("Labels exists:", labels.exists())

    if images.exists():
        print("Images:", len(list(images.glob("*"))))

    if labels.exists():
        print("Labels:", len(list(labels.glob("*.txt"))))

In [ ]:
# ============================================================
# CREATE SINGLE-CLASS BASELINE DATASET
# All original classes become:
# 0 = defect
#
# Images without annotations remain as background images.
# ============================================================

if BINARY_ROOT.exists():
    shutil.rmtree(BINARY_ROOT)

for split in ["train", "valid", "test"]:

    src_images = SOURCE_ROOT / split / "images"
    src_labels = SOURCE_ROOT / split / "labels"

    dst_images = BINARY_ROOT / split / "images"
    dst_labels = BINARY_ROOT / split / "labels"

    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    # Copy all images
    for image in src_images.iterdir():
        if image.is_file():
            shutil.copy2(image, dst_images / image.name)

    # Convert annotations
    for label_file in src_labels.glob("*.txt"):

        # Read annotation
        text = label_file.read_text().strip()

        # Empty annotation = background
        if not text:
            continue

        new_lines = []

        for line in text.splitlines():

            parts = line.split()

            if len(parts) != 5:
                continue

            # Ignore original class ID
            _, x, y, w, h = parts

            # Everything becomes class 0 = defect
            new_lines.append(
                f"0 {x} {y} {w} {h}"
            )

        if new_lines:
            output_file = dst_labels / label_file.name
            output_file.write_text("\n".join(new_lines))

print("Binary dataset created:")
print(BINARY_ROOT)

In [ ]:
# ============================================================
# VERIFY DATASET
# ============================================================

for split in ["train", "valid", "test"]:

    images = list((BINARY_ROOT / split / "images").glob("*"))
    labels = list((BINARY_ROOT / split / "labels").glob("*.txt"))

    image_names = {x.stem for x in images}
    label_names = {x.stem for x in labels}

    missing_labels = image_names - label_names

    print("\n" + "=" * 50)
    print(split.upper())
    print("=" * 50)

    print("Images:", len(images))
    print("Label files:", len(labels))
    print("Background/missing-label images:", len(missing_labels))

In [ ]:
# ============================================================
# CREATE YOLO DATASET YAML
# ============================================================

DATA_YAML = BINARY_ROOT / "data.yaml"

yaml_content = f"""path: {BINARY_ROOT}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: defect
"""

DATA_YAML.write_text(yaml_content)

print(DATA_YAML.read_text())

In [ ]:
# ============================================================
# LOAD YOLO11N
# ============================================================

model = YOLO("yolo11n.pt")

print("YOLO11n loaded successfully!")

In [ ]:
# ============================================================
# YOLO11 TRAINING
# ============================================================

train_results = model.train(
    data=str(DATA_YAML),

    epochs=50,
    imgsz=640,
    batch=16,

    device=0,

    patience=10,

    pretrained=True,

    project=str(RESULTS_ROOT),
    name="yolo11n_defect",

    plots=True,
    save=True,

    workers=2,

    seed=0,
    deterministic=True
)

print("Training completed!")

In [ ]:
# ============================================================
# LOCATE BEST MODEL
# ============================================================

BEST_MODEL = (
    RESULTS_ROOT /
    "yolo11n_defect" /
    "weights" /
    "best.pt"
)

print("Best model:", BEST_MODEL)
print("Exists:", BEST_MODEL.exists())

if BEST_MODEL.exists():
    print(
        f"Size: {BEST_MODEL.stat().st_size / (1024 * 1024):.2f} MB"
    )

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

shutil.copy2(
    BEST_MODEL,
    MODEL_OUTPUT
)

print("Final model saved:")
print(MODEL_OUTPUT)
print(
    f"Size: {MODEL_OUTPUT.stat().st_size / (1024 * 1024):.2f} MB"
)

In [ ]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

best_model = YOLO(str(MODEL_OUTPUT))

val_results = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)

print("Validation completed!")

In [ ]:
# ============================================================
# VALIDATION METRICS
# ============================================================

metrics = val_results.results_dict

precision = metrics.get("metrics/precision(B)", 0)
recall = metrics.get("metrics/recall(B)", 0)
map50 = metrics.get("metrics/mAP50(B)", 0)
map5095 = metrics.get("metrics/mAP50-95(B)", 0)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("VALIDATION RESULTS")
print("=" * 40)
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"mAP@50    : {map50:.4f}")
print(f"mAP@50-95 : {map5095:.4f}")

In [ ]:
# ============================================================
# SAVE VALIDATION LOG
# ============================================================

evaluation_data = {
    "model": ["YOLO11n"],
    "dataset": ["Mendeley Rail Defect"],
    "task": ["Binary Defect Detection"],
    "epochs": [50],
    "image_size": [640],
    "batch_size": [16],
    "precision": [precision],
    "recall": [recall],
    "f1_score": [f1],
    "mAP50": [map50],
    "mAP50_95": [map5095]
}

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_log = WORK_ROOT / "evaluation_log.csv"

evaluation_df.to_csv(
    evaluation_log,
    index=False
)

print("Saved:", evaluation_log)

In [ ]:
# ============================================================
# SAVE ULTRALYTICS TRAINING LOG
# ============================================================

TRAIN_CSV = (
    RESULTS_ROOT /
    "yolo11n_defect" /
    "results.csv"
)

TRAINING_LOG = WORK_ROOT / "training_log.csv"

if TRAIN_CSV.exists():

    df = pd.read_csv(TRAIN_CSV)

    df.to_csv(
        TRAINING_LOG,
        index=False
    )

    print("Training log saved:")
    print(TRAINING_LOG)

else:
    print("Training results CSV not found.")

In [ ]:
# ============================================================
# FINAL TRAINING SUMMARY
# ============================================================

print("=" * 60)
print("YOLO11 TRAINING COMPLETE")
print("=" * 60)

print("Model:")
print(MODEL_OUTPUT)

print("\nDataset:")
print(BINARY_ROOT)

print("\nTraining:")
print("Epochs     :", 50)
print("Image size :", 640)
print("Batch size :", 16)

print("\nValidation:")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1-score   : {f1:.4f}")
print(f"mAP@50     : {map50:.4f}")
print(f"mAP@50-95  : {map5095:.4f}")

print("\nYOLO11 model ready for independent testing.")